# 02 · El modelo y su entrenamiento

**Qué entra:** una ventana de 5 s de ECG (650 muestras a 130 Hz), ya filtrada y con z-score.

**Qué sale:** un vector de 128 números (la *huella*) de longitud 1. Dos ventanas de la misma persona deben dar vectores parecidos (coseno alto); de personas distintas, vectores distintos.

Este notebook muestra cada pieza funcionando. El fine-tuning se explica en el notebook 03.

In [ ]:
import sys, tempfile, dataclasses
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from src.config import Config, set_seed
from src.data import load_window_dir, split_polar
from src.synthetic import make_synthetic_table

set_seed(42)
polar_dir = ROOT / "data" / "processed" / "polar"
if list(polar_dir.glob("*.npz")):
    USING = "datos REALES del Polar H10"
    cfg = Config()
    table = load_window_dir(polar_dir)
    models_dir = ROOT / cfg.models_dir
else:
    USING = "datos SINTÉTICOS (aún no hay grabaciones en data/processed/polar)"
    cfg = dataclasses.replace(Config(), holdout_ids=("p4", "p5"), batch_size=32, patience=3,
                              polar_only_epochs=8, bootstrap_samples=200)
    table = make_synthetic_table(n_people=6, n_sessions=3, windows_per_session=40, seed=0)
    models_dir = Path(tempfile.mkdtemp())      # no ensuciamos models/ con pruebas
split = split_polar(table, cfg.holdout_ids)
print("Usando:", USING)
print({"train": len(split.train), "val": len(split.val), "hold-out": len(split.holdout)}, "ventanas")

## 1. Qué entra: ventanas de ECG
Cada persona tiene una forma de latido distinta (en datos reales, la forma depende de la anatomía, la posición del corazón y la colocación de la banda). Aquí vemos una ventana de tres personas.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
people = split.train.persons()[:3]
for ax, p in zip(axes, people):
    w = split.train.windows[split.train.participant_ids == p][0, 0]
    ax.plot(np.arange(len(w)) / cfg.fs, w)
    ax.set_ylabel(p)
axes[-1].set_xlabel("segundos")
fig.suptitle("Una ventana (z-score) de tres personas distintas")
plt.show()

## 2. La arquitectura
Un *stem* (convolución inicial), cuatro etapas residuales que van reduciendo la longitud a la mitad y aumentando los canales (32→64→128→128), un promedio en el tiempo y una capa lineal que produce el embedding. La *cabeza de clasificación* solo existe para entrenar y se descarta.

In [ ]:
from src.model import ECGEmbeddingNet
net = ECGEmbeddingNet(cfg, n_classes=len(split.train.persons()))
print(f"Parámetros: {net.n_parameters():,}")
x = torch.zeros(1, 1, cfg.window_samples)
h = net.stem(x); print("stem      ->", tuple(h.shape))
for i, s in enumerate(net.stages):
    h = s(h); print(f"etapa {i + 1}   ->", tuple(h.shape))
with torch.no_grad():
    z = net.eval()(x)
print("embedding ->", tuple(z.shape), "| norma:", float(z.norm()))

## 3. Entrenamiento (solo Polar, el modo de respaldo)
Se entrena como clasificador de las personas de entrenamiento. La parada temprana mira la **EER de validación** (sesión posterior de esas mismas personas), no la exactitud.

In [ ]:
from src.train import polar_only
import json
ckpt = polar_only(cfg, split, models_dir)
side = json.loads(ckpt.with_suffix(".json").read_text(encoding="utf-8"))
hist = side["metrics"]["history"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot([h["epoch"] for h in hist], [h["train_loss"] for h in hist]); ax[0].set_title("Pérdida de entrenamiento")
ax[1].plot([h["epoch"] for h in hist], [h["val_eer"] for h in hist]); ax[1].set_title("EER de validación")
for a in ax: a.set_xlabel("época")
plt.show()
print("Checkpoint:", ckpt.name, "| mejor EER de validación:", round(side["metrics"]["best_val_eer"], 3))

## 4. Cómo se ve la huella en 2D
Cada punto es una ventana proyectada de 128 a 2 dimensiones. Si la red funciona, las ventanas de una persona forman un grupo. **Los puntos de hold-out son personas que la red nunca vio.**

In [ ]:
from src.checkpoint import load_checkpoint
from src.evaluate import make_model_embed_fn
from src.data import concat_tables
trained = load_checkpoint(ckpt, cfg)
shown = concat_tables([split.val, split.holdout])
emb = make_model_embed_fn(trained)(shown.windows)
ids = shown.participant_ids
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for title, proj, a in (("PCA", PCA(2).fit_transform(emb), ax[0]),
                       ("t-SNE", TSNE(2, perplexity=min(30, len(emb) // 4), random_state=0).fit_transform(emb), ax[1])):
    for p in sorted(set(ids)):
        m = ids == p
        a.scatter(proj[m, 0], proj[m, 1], s=12, marker="x" if p in cfg.holdout_ids else "o",
                  label=p + (" (hold-out)" if p in cfg.holdout_ids else ""))
    a.set_title(title)
ax[1].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

## 5. Distribución de puntajes genuinos vs impostores (hold-out)
Puntaje = similitud coseno con la plantilla (registro con la 1.ª sesión). Mientras menos se traslapen las dos distribuciones, mejor separa el sistema a personas no vistas.

In [ ]:
from src.evaluate import split_enroll_test, run_protocol, eer, far_frr_at, first_session_table
embed = make_model_embed_fn(trained)
# El umbral se elige SOLO con validación (sesión posterior de las personas de entrenamiento);
# el hold-out jamás decide el umbral, solo se mide con él.
val_trials = run_protocol(embed, first_session_table(split.train), split.val)
val_eer, thr = eer(val_trials.genuine, val_trials.impostor)
enroll, test = split_enroll_test(split.holdout, "cross_session")
trials = run_protocol(embed, enroll, test)
hold_eer, _ = eer(trials.genuine, trials.impostor)
far, frr = far_frr_at(trials.genuine, trials.impostor, thr)
plt.hist(trials.impostor, bins=30, alpha=0.6, label="impostores")
plt.hist(trials.genuine, bins=30, alpha=0.6, label="genuinos")
plt.axvline(thr, color="k", ls="--", label=f"umbral (EER de validación={val_eer:.2f})")
plt.legend(); plt.xlabel("similitud coseno"); plt.title("Hold-out (personas no vistas)"); plt.show()
print(f"EER en hold-out: {hold_eer:.3f} | con el umbral de validación: FAR={far:.3f}, FRR={frr:.3f}")
print(f"Pares: {len(trials.genuine)} genuinos / {len(trials.impostor)} impostores de {trials.n_owners()} personas")